In [3]:
import pandas as pd
import numpy as np
# importer la base de données
df = pd.read_csv("dirty_sales_data.csv")
# aperçu rapide
display(df.head())

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clotting,$389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,$32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,TBD,Robert,Software,338.46,19
3,69c0e851-82fe-4794-9a2e-d561db153ca2,18/07/2025,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793 USD,15


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    530 non-null    str  
 1   Date              530 non-null    str  
 2   Client_Name       530 non-null    str  
 3   Product_Category  530 non-null    str  
 4   Unit_Price        530 non-null    str  
 5   Quantity          530 non-null    int64
dtypes: int64(1), str(5)
memory usage: 61.8 KB


In [5]:
# compter les doublons initiaux
print(f"Doublons : {df.duplicated().sum()}")
display(df[df.duplicated(keep=False)])

Doublons : 3


,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
30,548c30d1-1e62-424c-b096-315e9feb6d76,2025-12-25,Morin S.A.,Books,678.43,14
32,2f9b3255-41f3-4ea1-8bfb-9a60c6f63408,2025-07-08,Aubert,Home,1493.93,4
80,548c30d1-1e62-424c-b096-315e9feb6d76,2025-12-25,Morin S.A.,Books,678.43,14
90,713ea8f6-da8a-4f47-a45e-4b5a059cb2cd,2025-07-08,Chauveau,Home,692.48,1
293,2f9b3255-41f3-4ea1-8bfb-9a60c6f63408,2025-07-08,Aubert,Home,1493.93,4
435,713ea8f6-da8a-4f47-a45e-4b5a059cb2cd,2025-07-08,Chauveau,Home,692.48,1


In [7]:
# Supression des doublons exacts 
df = df.drop_duplicates()
print(f"Doublons après nettoyage : {df.duplicated().sum()}")

Doublons après nettoyage : 0


In [8]:
# Vérification des IDs dupliqués
duplicates_ids = df[df.duplicated(subset="Transaction_ID", keep=False)]
print(f"Nombre de lignes avec ID conflictuels : {len(duplicates_ids)}")
display(duplicates_ids.sort_values(by="Transaction_ID").head())

Nombre de lignes avec ID conflictuels : 53


,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
8,05556933-8214-495a-af5b-d6a007c2e6f7,2025-10-24,barre s.a.r.l.,Hom,533.02,95
259,05556933-8214-495a-af5b-d6a007c2e6f7,2025-04-28,Barre S.A.R.L.,Home,$533.02,6
107,161f5ac0-9e6f-4675-8c83-fc46450ccdcf,2025-07-26,Lecomte,Software,$408.93,12
190,161f5ac0-9e6f-4675-8c83-fc46450ccdcf,2026-01-10,Lecomte,Software,408.93,7
68,2685c5f7-2459-44eb-ba19-5cb6ea7582c9,2025-05-11,Aubert Regnier S.A.R.L.,Software,$293.71,4


In [9]:
# Stratégie : On garde la dernière occurence (règle métier)
# Sinon on ferait une investigation plus poussée pour comprendre pourquoi il y a des doublons sur les IDs
df = df.drop_duplicates(subset="Transaction_ID", keep="last")
df[df.duplicated(subset="Transaction_ID", keep=False)]

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity


In [10]:
# Aperçu du problème
print("Exemple de dates avant conversion :")
print(df["Date"])

Exemple de dates avant conversion :
0      2025-03-25
1      2026-01-19
2             TBD
3      18/07/2025
4      2025-10-31
          ...    
525    09/01/2026
526    18/11/2025
527    2025-02-12
528    2025-07-25
529    2025-08-12
Name: Date, Length: 500, dtype: str


In [11]:
# 1. Tentative Format ISO (YYYY-MM-DD)
dates_iso = pd.to_datetime(df["Date"], format="%Y-%m-%d", errors="coerce")
dates_iso.head(10)

0    2025-03-25
1    2026-01-19
2           NaT
3           NaT
4    2025-10-31
5    2025-08-18
6    2025-05-01
7    2025-03-05
9    2025-10-28
10   2025-06-23
Name: Date, dtype: datetime64[us]

In [12]:
# 2. Tentative Format FR (DD/MM/YYYY) sur les lignes en échec(Missing value)
mask_missed = dates_iso.isna()
dates_fr = pd.to_datetime(df.loc[mask_missed, "Date"], format="%d/%m/%Y", errors="coerce")
dates_fr.head(10)

2           NaT
3    2025-07-18
15   2025-09-21
25   2025-02-21
35   2025-08-09
36   2025-05-21
43   2025-06-18
49   2025-04-18
52   2025-10-15
57   2025-05-16
Name: Date, dtype: datetime64[us]

In [13]:
# 3. Combinaison : On remplit les trous de l'ISO avec les succès du FR
df["Date"] = dates_iso.fillna(dates_fr)
df.head(15)

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clotting,$389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,$32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,NaT,Robert,Software,338.46,19
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793 USD,15
5,af4b0483-1b41-4a4c-8fef-aa5f99a030e6,2025-08-18,Boyer Grégoire SA,Home,640.28,18
6,a0633138-65f5-4a43-89a7-20052a8debec,2025-05-01,Jacquot,Electornics,746.52,11
7,de210974-489d-499e-9408-94ff583b8c20,2025-03-05,roussel,Software,433.38,20
9,9dea572f-2bf0-47d9-84f4-b75625ce8f33,2025-10-28,Bodin,Electronics,1320.11,12
10,8f5e3a5c-4452-4d1c-81f7-10f32aca7f92,2025-06-23,Texier,Electronics,148 USD,4


In [16]:
# Bilan
saved_docs = dates_fr.notna().sum()
final_missing = df["Date"].isnull().sum()
print(f"\nDates récupérées via format FR : {saved_docs}")
print(f"Dates irrécupérables (Vraies poubelles) : {final_missing}")

# Supression des lignes sans date valide, indispensables pour la suite de analyse
df = df.dropna(subset=["Date"])
df.head(10)


Dates récupérées via format FR : 75
Dates irrécupérables (Vraies poubelles) : 0


,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clotting,$389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,$32.14,8
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793 USD,15
5,af4b0483-1b41-4a4c-8fef-aa5f99a030e6,2025-08-18,Boyer Grégoire SA,Home,640.28,18
6,a0633138-65f5-4a43-89a7-20052a8debec,2025-05-01,Jacquot,Electornics,746.52,11
7,de210974-489d-499e-9408-94ff583b8c20,2025-03-05,roussel,Software,433.38,20
9,9dea572f-2bf0-47d9-84f4-b75625ce8f33,2025-10-28,Bodin,Electronics,1320.11,12
10,8f5e3a5c-4452-4d1c-81f7-10f32aca7f92,2025-06-23,Texier,Electronics,148 USD,4
12,f28a7aaf-0096-4627-873b-4d1d515c9eec,2026-01-29,Renault,Electronics,"$1,453.80",8


In [17]:
df["Client_Name"]

0      De Oliveira Chartier S.A.R.L.
1                      Lebrun S.A.S.
3                             Pineau
4                              Leroy
5                  Boyer Grégoire SA
                   ...              
525                    Pichon S.A.S.
526                         Courtois
527                         Verdier 
528                         Maillard
529                            Roux 
Name: Client_Name, Length: 475, dtype: str

In [18]:
df["Client_Name"].str.strip()

0      De Oliveira Chartier S.A.R.L.
1                      Lebrun S.A.S.
3                             Pineau
4                              Leroy
5                  Boyer Grégoire SA
                   ...              
525                    Pichon S.A.S.
526                         Courtois
527                          Verdier
528                         Maillard
529                             Roux
Name: Client_Name, Length: 475, dtype: str

In [19]:
df["Client_Name"].str.strip().str.title()

0      De Oliveira Chartier S.A.R.L.
1                      Lebrun S.A.S.
3                             Pineau
4                              Leroy
5                  Boyer Grégoire Sa
                   ...              
525                    Pichon S.A.S.
526                         Courtois
527                          Verdier
528                         Maillard
529                             Roux
Name: Client_Name, Length: 475, dtype: str

In [21]:
# Nettoyage espace + Title Case 
df["Client_Name"] = df["Client_Name"].str.strip().str.title()

# Vérification rapide 
print(df["Client_Name"].sample(5))

91             Lambert Et Fils
346         Brunet Besson S.A.
441    Guillou Gautier Et Fils
48                    Bertrand
225                     Parent
Name: Client_Name, dtype: str


In [23]:
print("Valeurs uniques avant : ", df["Product_Category"].unique())

Valeurs uniques avant :  <ArrowStringArray>
[   'Clotting',    'Clothing',        'Home', 'Electronics', 'Electornics',
    'Software',       'Books',        'Book',        'H0me',        'Boks',
   'Softwaree',     'Sofware',         'Hom',     'Clothng',  'Electronic',
     'Softwar',        'Elec']
Length: 17, dtype: str


In [27]:
# Dictionnaire de correction 

corrections = {
    "Software" : "Software", "Softwaree" : "Software", "Softwar" : "Software", "Sofware" : "Software",
    "Clotting" : "Clothing", "Clothing" : "Clothing", "Clothng" : "Clothing",
    "Home" : "Home", "H0me" : "Home", "Hom" : "Home", 
    "Electronics" : "Electronics", "Electornics" : "Electronics", "Electronic" : "Electronics",
    "Elec" : "Electronics", 
    "Book" : "Books", "Boks" : "Books"
}

# Application (replace ou map)
df["Product_Category"] = df["Product_Category"].replace(corrections)
print("\nValeurs uniques après : ", df["Product_Category"].unique())


Valeurs uniques après :  <ArrowStringArray>
['Clothing', 'Home', 'Electronics', 'Software', 'Books']
Length: 5, dtype: str


In [28]:
df["Unit_Price"].head()

0    $389.54
1     $32.14
3     544.11
4    793 USD
5     640.28
Name: Unit_Price, dtype: str

In [29]:
"$793 USD".replace("$", "")

'793 USD'

In [30]:
"$793 USD ".replace("$", "").replace("USD", "").strip()

'793'

In [37]:
p = "1,000,000.73"
p = "1.000.00,73"
p = "10000,73"
if "," in p and "." in p :
    if p.rfind(",") > p.rfind("."):
        # Format US 1,000.00 mais inversé ?? Ou juste Europe 1.000,00
        # On remplace "." par rien, et "," par "."
        p = p.replace(".", "").replace(",", ".").strip()
    else : 
        # Format US 1,000.00  -> remove
        p = p.replace(",", "")
elif "," in p:
    # Juste une virgule ? Probablement décimale si pas de point 
    p = p.replace(",", ".")    

p       

'10000.73'

In [39]:
def clean_price(price_str):
    if isinstance(price_str, (int, float)):
        return float(price_str)
    # Convertir en string propre 
    p = str(price_str)

    # Cas simple : On retire devises et espaces     
    p = p.replace("$", "").replace("€", "").replace("USD", "").strip()

    # Gestion format Europe (point pour millier, virgule pour décimale)
    # Heuristique simple : si "," est le dernier séparateur, c'est une décimale
    if "," in p and "." in p : 
        if p.rfind(",") > p.rfind("."):
            # Format US 1,000.00 mais inversé ?? Ou juste Europe 1.000,00
            # On remplace "." par rien, et "," par "."
            p = p.replace(".", "").replace(",", ".")
        else : 
            # Format US 1,000.00 -> remove
            p = p.replace(",", "")
    elif "," in p: 
        # Juste une virgule ? Probablement décimale si pas de point 
        p = p.replace(",", ".")

    return float(p)    
df["Unit_Price"] = df["Unit_Price"].apply(clean_price)

display(df.head(10))    

print(df["Unit_Price"].describe())


,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clothing,389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,32.14,8
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793.00,15
5,af4b0483-1b41-4a4c-8fef-aa5f99a030e6,2025-08-18,Boyer Grégoire Sa,Home,640.28,18
6,a0633138-65f5-4a43-89a7-20052a8debec,2025-05-01,Jacquot,Electronics,746.52,11
7,de210974-489d-499e-9408-94ff583b8c20,2025-03-05,Roussel,Software,433.38,20
9,9dea572f-2bf0-47d9-84f4-b75625ce8f33,2025-10-28,Bodin,Electronics,1320.11,12
10,8f5e3a5c-4452-4d1c-81f7-10f32aca7f92,2025-06-23,Texier,Electronics,148.00,4
12,f28a7aaf-0096-4627-873b-4d1d515c9eec,2026-01-29,Renault,Electronics,1453.80,8


count     475.000000
mean      742.893663
std       433.771543
min        14.840000
25%       376.495000
50%       737.910000
75%      1110.125000
max      1499.070000
Name: Unit_Price, dtype: float64


In [40]:
print("Avant nettoyage :")
print(df["Quantity"].describe())

Avant nettoyage :
count     475.000000
mean       65.416842
std       568.909846
min        -5.000000
25%         6.000000
50%        11.000000
75%        16.000000
max      9206.000000
Name: Quantity, dtype: float64


In [41]:
# Règles métiers : 
# 1. Quantity > 0
# 2. Quantity < 100 (seuil arbitraire)

condition_valide = (df["Quantity"] > 0) & (df["Quantity"] < 100)
condition_valide


0      True
1      True
3      True
4      True
5      True
       ... 
525    True
526    True
527    True
528    True
529    True
Name: Quantity, Length: 475, dtype: bool

In [42]:
df_clean = df[condition_valide].copy()
df_clean

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clothing,389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,32.14,8
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793.00,15
5,af4b0483-1b41-4a4c-8fef-aa5f99a030e6,2025-08-18,Boyer Grégoire Sa,Home,640.28,18
...,...,...,...,...,...,...
525,b9043d66-57df-421b-b267-b33a630b7e49,2026-01-09,Pichon S.A.S.,Electronics,100.31,11
526,9087f5dc-b05b-4281-a5a0-123e2afc873e,2025-11-18,Courtois,Clothing,404.05,4
527,c5d98856-bce1-44de-b705-b210ac8a9e6a,2025-02-12,Verdier,Books,995.16,7
528,c8c06140-d548-4021-85ee-a70b9db69f5d,2025-07-25,Maillard,Electronics,278.82,10


In [43]:
print("\nAprès nettoyage :")
print(df_clean["Quantity"].describe())


Après nettoyage :
count    461.000000
mean      11.713666
std        8.430561
min        1.000000
25%        7.000000
50%       11.000000
75%       16.000000
max       88.000000
Name: Quantity, dtype: float64


In [44]:
df_clean.to_csv("cleaned_sales_data.csv", index=False)
print("\nFichier nettoyé sauvegardé sous : cleaned_sales_data.csv")


Fichier nettoyé sauvegardé sous : cleaned_sales_data.csv
